<a href="https://colab.research.google.com/github/srinivasveeru/Training/blob/main/BronzeAssignmentProjectWithRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Use case: “Policy & Claims Copilot” (Customer support + Claims pre-check) using RAG
#Steps cover Loading document, Creating Chunks, Embedding, and storing in vector database
!pip install langchain langchain-community langchain-openai pypdf chromadb faiss-cpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142

In [2]:
#Code to load the pdf file provided as input
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('InsurancePolicy.pdf')
docpdf = loader.load()

print(f"From PDF {len(docpdf)} pages are loaded.")
#for page in range(len(docpdf)):
  #print(docpdf[page])

From PDF 27 pages are loaded.


In [3]:
#code to create chunks - chunk size with overlap info will specify how many chunks are created
from langchain_text_splitters import RecursiveCharacterTextSplitter

textsplit = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = textsplit.split_documents(docpdf)

print(f"Document split into {len(chunks)} chunks.")
#for i in range(len(chunks)):
  #print(f"chunk number: {i}\n")
  #print(chunks[i])

Document split into 125 chunks.


In [4]:
#code to create embeddings and store in vector database
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import os

#API Key to be obtained from secret key in Google colab
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

#Embedding call and storing chunks in vector database
embeddings = OpenAIEmbeddings()
vector_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

print("Vector store created and populated.")

Vector store created and populated.


In [5]:
#Code to open LLM used for this exercise and create code to retrieve content
from langchain_classic.chains import RetrievalQA
from langchain_classic.chat_models import ChatOpenAI

# Initialize the Language Model (e.g., GPT-3.5-turbo)
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# Create a retrieval chain
qa_chain = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)


/tmp/ipykernel_1071/611475743.py:6: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-4o", temperature=0)


In [6]:
#code created for any queries from user pertaining to the document
query = input()
response = qa_chain.run(query)
print(response)

provide claim process


/tmp/ipykernel_1071/2335806927.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa_chain.run(query)


The claim process involves several steps, depending on whether you are opting for reimbursement or cashless claims. Here's a summary of the procedures:

**Claims Procedure for Reimbursement:**
1. **Consultation and Treatment:** The insured should consult a doctor without delay, follow the recommended treatment, and take steps to minimize the claim amount.
2. **Intimation to Insurer:** Notify the insurer immediately, and in any case, within 48 hours from the date of hospitalization. The insurer may allow a delay up to 7 days if a justifiable reason is provided.
3. **Submission of Claim Documents:** File the claim with all necessary documentation within 15 days of discharge. Required documents include:
   - Duly filled claim form
   - Valid photo identity card
   - Original discharge card/certificate/death summary
   - Copies of prescriptions for diagnostic tests, treatment advice, and medical references
   - Original set of investigation reports
   - Itemized original hospital bill and 